# [ 실습4 ] HuggingFace 첫 감정분석 모델

> AI기반 실시간 대화 분석 및 맥락정리 기술개발 R&D — HuggingFace 입문

## 실습 목표

1. `pipeline()` API로 HuggingFace 첫 모델 동작 확인
2. 모델 내부 구조 살펴보기 (DistilBERT 아키텍처)
3. Sarcasm / 혼합 감정으로 모델 한계 체감
4. 한국어 감정분석 모델로 교체
5. 회의 데이터에 적용 — 강요된 동의 컨셉 연결

## 환경

| 항목 | 버전 |
|------|------|
| Python | 3.11.15 |
| transformers | 5.9.0 |
| torch | 2.11.0+cu128 |
| GPU | RTX 5070 (Blackwell, sm_120) |

transformers 5.9.0 검증 완료. `pipeline()` API는 4.x에서 5.x로 넘어오면서 기본 호출 방식은 변화가 거의 없음. 단, `device="cuda"` 명시 필요 (자동 GPU 할당 안 됨).

## 연구 컨텍스트

XR 환경 다자간 대화에서 **표면과 속내의 괴리** 탐지가 최종 목표.

- "네 좋아요" + 심박 스트레스 → 강요된 동의
- 칭찬 + 팔짱 + 눈 굴림 → Sarcasm

본 실습의 감정분석은 텍스트 모달리티의 출발점에 해당.

In [2]:
# 환경 확인 — 이 노트북이 어떤 환경에서 실행됐는지 영구 기록
import sys
import torch
import transformers

print(f"Python:       {sys.version.split()[0]}")
print(f"transformers: {transformers.__version__}")
print(f"torch:        {torch.__version__}")
print(f"CUDA 사용:    {torch.cuda.is_available()}")
print(f"GPU:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

Python:       3.11.15
transformers: 5.9.0
torch:        2.11.0+cu128
CUDA 사용:    True
GPU:          NVIDIA GeForce RTX 5070


In [3]:
# HuggingFace pipeline 생성
# task: sentiment-analysis (text-classification의 별칭)
# device: cuda → RTX 5070 사용 (명시 안 하면 CPU로 돌아감)
from transformers import pipeline # HuggingFace의 transformers 라이브러리에서 pipeline이라는 함수를 가져옴

classifier = pipeline("sentiment-analysis", device="cuda")
# pipeline()의 첫 번째 인자: task: 어떤 모델을 가져올지 명시(위 코드는 감정분석 모델)
# cuda: nvida gpu -> 이것을 명시해야 GPU를 사용
# 변환된 객체를 classifier 변수에 저장

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

모델을 명시하지 않으면 기본값을 골라줌.

실제 운용/논문 코드에서는 명시해야 할듯.

```python
pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    revision="714eb0f",   # 특정 버전 고정
    device="cuda",
)
```

In [6]:
# 첫 추론 - 영어 한 문장을 분류해보기
result = classifier("I love this movie!")
print(result)

# 이렇게 classifier()을 한 번 호출하면
# 1: 토크나이저가 문자열 -> 토큰 ID 리스트로 변환
# 2: 토큰 ID를 텐서로 변환 후 GPU 이동
# 3: DistilBERT 모델이 forward pass 실행(6개의 transformer 레이어 통과)
# 4: 마지막 토큰 출력에서 분류 점수 계산(ex..POSITIVE = 0.9998775720596313 / NEGATIVE = 0.0001224279403687 에서 큰 거 선택)
# 5: softmax로 확률화 -> 가장 높은 라벨 선택
# 6: 사람이 읽기 좋은 dict 형태로 출력

[{'label': 'POSITIVE', 'score': 0.9998775720596313}]


In [ ]:
# 여러 문장을 한 번에 분류 - 모델의 패턴과 한계 동시 관찰
sentences = [
    "I love this movie!",                              # 명백한 긍정
    "This is the worst film ever.",                    # 명백한 부정
    "It was okay, I guess.",                           # 애매한 중립
    "Oh great, another Monday morning.",               # Sarcasm (비꼼) -> 사실은 부정인데 긍정처럼 보이는 문장
    "The food was amazing but service was terrible.",  # 혼합 감정
]

results = classifier(sentences)

# 그 결과를 보기 좋게 출력
for sentence, result in zip(sentences, results):
    label = result['label']
    score = result['score']
    print(f"[{label:8s} {score:.4f}]  {sentence}")

# 5개 문장 입력하면 토크나이저가 5개 모두 토큰화 + 패딩(길이 맞춤)
# GPU로 5개를 한 번에 전송
# DistilBERT가 5개를 병렬 처리(GPU 장점) 후 5개 결과를 동시에 받음 -> GPU는 병렬 연산에 강해서 입력 수가 늘어나도 처리 시간이 비례해서 늘어나지 않음

[POSITIVE 0.9999]  I love this movie!
[NEGATIVE 0.9997]  This is the worst film ever.
[POSITIVE 0.9997]  It was okay, I guess.
[POSITIVE 0.9983]  Oh great, another Monday morning.
[POSITIVE 0.6300]  The food was amazing but service was terrible.


In [10]:
# 결과 해석
# - 지금 사용 중인 모델: distilbert/distilbert-base-uncased-finetuned-sst-2-english -> 이진 분류 모델
# - 3번째 결과 보면 "애매한 중립" 문장을 강력한 긍정 문장으로 파악 -> 이진 분류의 한계점(중립 옵션이 없음 + 높은 홧신으로 답함)
# - 4번째 결과 보면 사회적인 의미로 부정 문장인데 다눈히 문장 속 단어만 파악하고 긍정 문장으로 파악 -> 즉 이러한 한계점이 바로 멀티모달이 필요한 이유
# - 5번째 결과 보면 이전까지는 모두 높은 score 점수로 예측을 했는데 0.6300이라는 낮은 스코어로 예측 -> 모델이 헷갈렸다는 징조

In [15]:
# 챌린지 2: 영어 Sarcasm 명장 3개 만들기
# 의도: 표면 단어와 실제 의미의 괴리를 만들어서 모델이 속는지 확인

# 내가 생성한 문장
my_sarcasm = [
    "Wonderful, my flight got delayed by 5hours",
    "What a brilliant idea to schedule a meeting at 6 AM",
    "Just what I needed, more emails to answer",
]

# 분류 실행
my_results = classifier(my_sarcasm)

# 결과 출력
for sentence, result in zip(my_sarcasm, results):
    label = result['label']
    score = result['score']
    print(f"{label:8s} {score:.4f} {sentence}")

POSITIVE 0.9999 Wonderful, my flight got delayed by 5hours
NEGATIVE 0.9997 What a brilliant idea to schedule a meeting at 6 AM
POSITIVE 0.9997 Just what I needed, more emails to answer
